In [3]:
import os
import glob
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import esmpy
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.ticker as ticker

In [7]:
grid_src = esmpy.Grid(filename='era5/src_grid.nc', filetype=esmpy.constants.FileFormat.SCRIP, is_sphere=False)
grid_dst = esmpy.Grid(filename='era5/dst_grid.nc', filetype=esmpy.constants.FileFormat.SCRIP, is_sphere=False)

# 1h -> 96
#field_src = esmpy.Field(grid_src, staggerloc=esmpy.StaggerLoc.CENTER, ndbounds=[96,])
#field_dst = esmpy.Field(grid_dst, staggerloc=esmpy.StaggerLoc.CENTER, ndbounds=[96,])

# 6h -> 16
field_src = esmpy.Field(grid_src, staggerloc=esmpy.StaggerLoc.CENTER, ndbounds=[16,])
field_dst = esmpy.Field(grid_dst, staggerloc=esmpy.StaggerLoc.CENTER, ndbounds=[16,])

regrid_src2dst = esmpy.Regrid(
    field_src,
    field_dst,
    regrid_method=esmpy.RegridMethod.BILINEAR,
    unmapped_action=esmpy.UnmappedAction.IGNORE
)

In [8]:
grid = xr.open_dataset('../run/Data/ROMS/irene_roms_grid.nc')

In [9]:
# Load in dataset
ds_era5 = xr.open_dataset('era5/data_stream-oper_stepType-accum.nc')

# Create dataset to store interpolated values
ds_era5_interp = xr.Dataset()

for v in ds_era5.data_vars.keys():
    print(v)
    # Interpolate field
    field_dst.data[:] = 1e20
    field_src.data[:] = ds_era5[v].transpose().values
    field_dst = regrid_src2dst(field_src, field_dst, zero_region=esmpy.Region.SELECT)

    # Add result to new dataset
    ds_era5_interp[v] = xr.DataArray(
        field_dst.data[:].transpose().copy(),
        dims = {
            'time': ds_era5.valid_time.values,
            'eta_rho': grid.eta_rho,
            'xi_rho': grid.xi_rho,
        }
    )
    # Copy attributes
    ds_era5_interp[v].attrs.update(ds_era5[v].attrs)

# Add time variable
ds_era5_interp['time'] = xr.DataArray(
    ds_era5['valid_time'].values,
    dims = {
        'time': ds_era5.valid_time.values
    }
)
ds_era5_interp['time'].attrs.update(ds_era5['valid_time'].attrs)

# Write new dataset to disk
ds_era5_interp.to_netcdf(f"era5/data_stream-oper_stepType-accum-interp.nc")

tp
slhf
ssr
str
sshf
ssrd
strd


In [10]:
# Load in dataset
ds_era5 = xr.open_dataset('era5/data_stream-oper_stepType-instant.nc')

# Create dataset to store interpolated values
ds_era5_interp = xr.Dataset()

for v in ds_era5.data_vars.keys():
    print(v)
    # Interpolate field
    field_dst.data[:] = 1e20
    field_src.data[:] = ds_era5[v].transpose().values
    field_dst = regrid_src2dst(field_src, field_dst, zero_region=esmpy.Region.SELECT)

    # Add result to new dataset
    ds_era5_interp[v] = xr.DataArray(
        field_dst.data[:].transpose().copy(),
        dims = {
            'time': ds_era5.valid_time.values,
            'eta_rho': grid.eta_rho,
            'xi_rho': grid.xi_rho,
        }
    )
    # Copy attributes
    ds_era5_interp[v].attrs.update(ds_era5[v].attrs)

# Add time variable
ds_era5_interp['time'] = xr.DataArray(
    ds_era5['valid_time'].values,
    dims = {
        'time': ds_era5.valid_time.values
    }
)
ds_era5_interp['time'].attrs.update(ds_era5['valid_time'].attrs)

# Write new dataset to disk
ds_era5_interp.to_netcdf(f"era5/data_stream-oper_stepType-instant-interp.nc")

u10
v10
d2m
t2m
msl
sst
